In [ ]:
# imports

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# paths and constants

PROCESSED_DIR = Path("processed_data")
FIGURE_DIR = Path("figures") / "eda"

PROCESSED_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SCADA_CLEAN_FILE = PROCESSED_DIR / "penmanshiel_scada_clean.parquet"
SCADA_COMPLETE_FILE = PROCESSED_DIR / "penmanshiel_scada_complete.parquet"

ROW_FLAGS_FILE = PROCESSED_DIR / "penmanshiel_curtailment_flags.parquet"
EXCLUDED_TIMESTAMPS_FILE = (
    PROCESSED_DIR / "penmanshiel_excluded_curtailment_timestamps.parquet"
)
SCADA_FLAGGED_FILE = (
    PROCESSED_DIR / "penmanshiel_scada_complete_with_curtailment_flags.parquet"
)

RATED_POWER_KW = 2050


In [ ]:
# detect curtailment-like operation

def detect_curtailment(
    scada,
    rated_power_kw=2050,
    window_steps=6,
    window_steps_wind=4,
    flat_std_max_kw=20,
    power_min_kw=100,
    power_max_fraction=0.95,
    deficit_fraction=0.85,
    ws_range_min=1.5,
    flat_wind_power_max_fraction=0.97,
    pitch_threshold_deg=30,
    pitch_wind_min=3.5,
    pitch_power_max_fraction=0.05,
    min_flagged_turbines=1,
    expand_steps=1,
):
    data = scada.copy()

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True,
    )

    data = data.sort_values(
        ["turbine_id", "timestamp"]
    )

    # flag explicit curtailment and pitch-stopped operation

    data["max_pitch"] = data[
        ["pitch_a", "pitch_b", "pitch_c"]
    ].max(axis=1)

    data["pitch_stopped"] = (
        (data["max_pitch"] > pitch_threshold_deg)
        & (data["wind_speed"] >= pitch_wind_min)
        & (
            data["power_kw"]
            <= pitch_power_max_fraction * rated_power_kw
        )
    )

    data["explicit_curtailed"] = (
        data["curtailment_kwh"].fillna(0) > 0
    )

    # build a reference power curve from rows without obvious restrictions

    reference_source = data[
        ~data["pitch_stopped"]
        & ~data["explicit_curtailed"]
        & ~(
            (data["wind_speed"] > 4)
            & (data["power_kw"] < 50)
        )
    ].copy()

    wind_speed_bins = np.arange(
        0,
        30.5,
        0.5,
    )

    reference_curve = (
        reference_source
        .groupby(
            pd.cut(
                reference_source["wind_speed"],
                wind_speed_bins,
            ),
            observed=True,
        )["power_kw"]
        .median()
        .dropna()
    )

    reference_bin_centres = np.array(
        [
            interval.mid
            for interval in reference_curve.index
        ]
    )

    data["expected_power_kw"] = np.interp(
        np.clip(
            data["wind_speed"],
            0,
            30,
        ),
        reference_bin_centres,
        reference_curve.values,
    )

    # flag sustained flat power within each turbine series

    flat_flag_parts = []

    for turbine_id, turbine_data in data.groupby("turbine_id"):
        turbine_data = (
            turbine_data
            .set_index("timestamp")
            .sort_index()
        )

        complete_grid = pd.date_range(
            turbine_data.index.min().floor("10min"),
            turbine_data.index.max().ceil("10min"),
            freq="10min",
            tz="UTC",
        )

        turbine_grid = turbine_data[
            [
                "power_kw",
                "expected_power_kw",
                "wind_speed",
            ]
        ].reindex(complete_grid)

        power_std = (
            turbine_grid["power_kw"]
            .rolling(
                window_steps,
                min_periods=window_steps,
            )
            .std()
        )

        power_mean = (
            turbine_grid["power_kw"]
            .rolling(
                window_steps,
                min_periods=window_steps,
            )
            .mean()
        )

        expected_power_mean = (
            turbine_grid["expected_power_kw"]
            .rolling(
                window_steps,
                min_periods=window_steps,
            )
            .mean()
        )

        deficit_flat = (
            (power_std <= flat_std_max_kw)
            & (power_mean >= power_min_kw)
            & (
                power_mean
                <= power_max_fraction * rated_power_kw
            )
            & (
                power_mean
                <= deficit_fraction * expected_power_mean
            )
        )

        short_power_std = (
            turbine_grid["power_kw"]
            .rolling(
                window_steps_wind,
                min_periods=window_steps_wind,
            )
            .std()
        )

        short_power_mean = (
            turbine_grid["power_kw"]
            .rolling(
                window_steps_wind,
                min_periods=window_steps_wind,
            )
            .mean()
        )

        wind_speed_range = (
            turbine_grid["wind_speed"]
            .rolling(
                window_steps_wind,
                min_periods=window_steps_wind,
            )
            .max()
            - turbine_grid["wind_speed"]
            .rolling(
                window_steps_wind,
                min_periods=window_steps_wind,
            )
            .min()
        )

        flat_vs_wind = (
            (short_power_std <= flat_std_max_kw)
            & (wind_speed_range >= ws_range_min)
            & (short_power_mean >= power_min_kw)
            & (
                short_power_mean
                <= flat_wind_power_max_fraction
                * rated_power_kw
            )
        )

        def spread_flag_backwards(end_flag, steps):
            return (
                end_flag[::-1]
                .rolling(
                    steps,
                    min_periods=1,
                )
                .max()[::-1]
                .astype(bool)
            )

        flat_curtailed = (
            spread_flag_backwards(
                deficit_flat,
                window_steps,
            )
            | spread_flag_backwards(
                flat_vs_wind,
                window_steps_wind,
            )
        )

        flat_flag_parts.append(
            pd.DataFrame(
                {
                    "timestamp": complete_grid,
                    "turbine_id": turbine_id,
                    "flat_curtailed": flat_curtailed.values,
                }
            )
        )

    flat_flags = pd.concat(
        flat_flag_parts,
        ignore_index=True,
    )

    data = data.merge(
        flat_flags,
        on=["timestamp", "turbine_id"],
        how="left",
    )

    data["flat_curtailed"] = (
        data["flat_curtailed"]
        .fillna(False)
        .astype(bool)
    )

    data["row_curtailed"] = (
        data["flat_curtailed"]
        | data["pitch_stopped"]
        | data["explicit_curtailed"]
    )

    # convert row-level flags to the farm-level exclusion timestamps

    flagged_turbines_by_timestamp = (
        data
        .groupby("timestamp")["row_curtailed"]
        .sum()
    )

    timestamp_flag = (
        flagged_turbines_by_timestamp
        >= min_flagged_turbines
    )

    complete_time_grid = pd.date_range(
        timestamp_flag.index.min().floor("10min"),
        timestamp_flag.index.max().ceil("10min"),
        freq="10min",
        tz="UTC",
    )

    timestamp_flag = timestamp_flag.reindex(
        complete_time_grid,
        fill_value=False,
    )

    expanded_flag = timestamp_flag.copy()

    for step in range(1, expand_steps + 1):
        expanded_flag = (
            expanded_flag
            | timestamp_flag.shift(
                step,
                fill_value=False,
            )
            | timestamp_flag.shift(
                -step,
                fill_value=False,
            )
        )

    observed_timestamps = pd.DatetimeIndex(
        data["timestamp"].unique()
    )

    excluded_timestamps = (
        expanded_flag.index[expanded_flag]
        .intersection(observed_timestamps)
    )

    row_flags = data[
        [
            "timestamp",
            "turbine_id",
            "row_curtailed",
            "flat_curtailed",
            "pitch_stopped",
            "explicit_curtailed",
        ]
    ].copy()

    return row_flags, excluded_timestamps


In [ ]:
# run the detection on the cleaned turbine-level dataset

scada_clean = pd.read_parquet(
    SCADA_CLEAN_FILE
).copy()

row_flags, excluded_timestamps = detect_curtailment(
    scada_clean,
    rated_power_kw=RATED_POWER_KW,
)

flag_summary = pd.Series(
    {
        "flat-streak rows": row_flags["flat_curtailed"].sum(),
        "pitch-stopped rows": row_flags["pitch_stopped"].sum(),
        "explicit curtailment rows": row_flags["explicit_curtailed"].sum(),
        "curtailment-like rows": row_flags["row_curtailed"].sum(),
        "excluded farm timestamps": len(excluded_timestamps),
    }
)

print(flag_summary.to_string())

print(
    f"curtailment-like rows: "
    f"{100 * row_flags['row_curtailed'].mean():.2f}%"
)

print(
    f"excluded farm timestamps: "
    f"{100 * len(excluded_timestamps) / scada_clean['timestamp'].nunique():.2f}%"
)


In [ ]:
# attach the flags to the complete 14-turbine dataset

scada_complete = pd.read_parquet(
    SCADA_COMPLETE_FILE
).copy()

scada_complete["timestamp"] = pd.to_datetime(
    scada_complete["timestamp"],
    utc=True,
)

scada_flagged = scada_complete.merge(
    row_flags,
    on=["timestamp", "turbine_id"],
    how="left",
)

flag_columns = [
    "row_curtailed",
    "flat_curtailed",
    "pitch_stopped",
    "explicit_curtailed",
]

scada_flagged[flag_columns] = (
    scada_flagged[flag_columns]
    .fillna(False)
    .astype(bool)
)


In [ ]:
# save the curtailment flags and farm-level exclusion timestamps

row_flags.to_parquet(
    ROW_FLAGS_FILE,
    index=False,
)

pd.DataFrame(
    {"timestamp": excluded_timestamps}
).to_parquet(
    EXCLUDED_TIMESTAMPS_FILE,
    index=False,
)

scada_flagged.to_parquet(
    SCADA_FLAGGED_FILE,
    index=False,
)

print(f"saved {ROW_FLAGS_FILE}")
print(f"saved {EXCLUDED_TIMESTAMPS_FILE}")
print(f"saved {SCADA_FLAGGED_FILE}")


In [ ]:
# figure 3.10 curtailment-like operation for wt08 in 2016

plot_turbine = 8
plot_year = 2016

plot_data = scada_flagged[
    (scada_flagged["turbine_id"] == plot_turbine)
    & (
        scada_flagged["timestamp"].dt.year
        == plot_year
    )
].copy()

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    plot_data.loc[
        ~plot_data["row_curtailed"],
        "wind_speed",
    ],
    plot_data.loc[
        ~plot_data["row_curtailed"],
        "power_kw",
    ],
    s=5,
    alpha=0.15,
    color="steelblue",
    label="Not flagged",
)

ax.scatter(
    plot_data.loc[
        plot_data["row_curtailed"],
        "wind_speed",
    ],
    plot_data.loc[
        plot_data["row_curtailed"],
        "power_kw",
    ],
    s=6,
    alpha=0.55,
    color="orange",
    label="Curtailment-like",
)

ax.set_xlabel(
    "Wind speed (m/s)",
    fontsize=14,
)

ax.set_ylabel(
    "Power (kW)",
    fontsize=14,
)

ax.set_xlim(0, 25)
ax.set_ylim(0, 2200)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("black")

ax.tick_params(
    axis="both",
    direction="out",
    length=5,
    width=1,
    labelsize=12,
    top=False,
    right=False,
)

ax.grid(
    alpha=0.20,
    linewidth=0.8,
)

ax.set_axisbelow(True)

ax.legend(
    frameon=False,
    fontsize=10,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "fig_3_10_curtailment_detection.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
